In [0]:
# Notebook role :
# Clean data
# Remove nulls
# Deduplicate
# Save Silver Delta tables


In [0]:
silver_path = "abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/"

In [0]:
spark.conf.set(
  "fs.azure.account.key.mystorage.dfs.core.windows.net",
  "ACCESS_KEY_REMOVED"
)

In [0]:
bronze_path = "abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/bronze/"

df_acct = spark.read.format("delta").load(bronze_path + "accounts")
df_cust = spark.read.format("delta").load(bronze_path + "customers")
df_loan = spark.read.format("delta").load(bronze_path + "loans")
df_tran = spark.read.format("delta").load(bronze_path + "transactions")
df_ledg = spark.read.format("delta").load(bronze_path + "ledger")

print("Bronze tables loaded successfully")

Bronze tables loaded successfully


In [0]:
# Cleaning Accounts DF
df_acct_clean = df_acct.filter(df_acct.balance.isNotNull())
df_acct_clean = df_acct_clean.dropDuplicates()

df_acct_clean.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path + "accounts")

print("Accounts cleaned and saved successfully")

Accounts cleaned and saved successfully


In [0]:
display(dbutils.fs.ls(silver_path))

path,name,size,modificationTime
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/accounts/,accounts/,0,1772370894000


In [0]:
# Cleaning the Customers Table 
#  1. remove duplicate customers 
#  2. Ensure customer_id is not null 

from pyspark.sql.functions import col

df_cust_clean = (
    df_cust
    .filter(col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

df_cust_clean.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path + "customers")

print("Customers cleaned and saved successfully")

Customers cleaned and saved successfully


In [0]:
# Clean the Loans Table
# 1. Remove null loan_id
# 2. Remove negative or zero loan amounts
# 3. remove duplicates
df_loan_clean = (
    df_loan
    .filter(col("loan_id").isNotNull())
    .filter(col("principal_amount") > 0)
    .dropDuplicates(["loan_id"])
)

df_loan_clean.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path + "loans")

print("Loans cleaned and saved successfully")


Loans cleaned and saved successfully


In [0]:
# Cleaning the Transactions table .
# Remove null transaction_id
# Remove zero or negative amounts
# Remove null status
# Remove duplicates

df_tran_clean = (
    df_tran
    .filter(col("transaction_id").isNotNull())
    .filter(col("amount") > 0)
    .filter(col("tran_status").isNotNull())
    .dropDuplicates(["transaction_id"])
)

df_tran_clean.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path + "transactions")

print("Transactions cleaned and saved successfully")

Transactions cleaned and saved successfully


In [0]:
display(dbutils.fs.ls(silver_path))

path,name,size,modificationTime
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/accounts/,accounts/,0,1772370894000
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/customers/,customers/,0,1772371140000
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/loans/,loans/,0,1772371410000
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/transactions/,transactions/,0,1772371572000


In [0]:
# clean Ledger Table 
df_ledg_clean = (
    df_ledg
    .dropDuplicates()
)

df_ledg_clean.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path + "ledger")

print("Ledger cleaned and saved successfully")

Ledger cleaned and saved successfully


In [0]:
display(dbutils.fs.ls(silver_path))

path,name,size,modificationTime
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/accounts/,accounts/,0,1772370894000
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/customers/,customers/,0,1772371140000
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/ledger/,ledger/,0,1772371638000
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/loans/,loans/,0,1772371410000
abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/transactions/,transactions/,0,1772371572000
